In [51]:
import pandas as pd
import openpyxl
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [52]:
def process_te_2025():
    """
    Replica exatamente o Power Query M:
    1. Lê Sheet1
    2. Promove headers (pega linha de título)
    3. Skip 1 linha
    4. Promove headers NOVAMENTE (pega o header real)
    5. Aplica transformações
    """
    
    file_path = r".\T&E_Concur_Imaging.xlsx"
    
    # Ler Sheet1 sem header
    df = pd.read_excel(file_path, sheet_name="Sheet1", header=None)
    
    print(f"[DEBUG] Shape original: {df.shape}")
    
    # PASSO 1: Promover primeira linha como header (igual Power M faz primeiro)
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    
    print(f"[DEBUG] Após 1º PromoteHeaders: colunas={df.columns.tolist()[:5]}")
    
    # PASSO 2: Skip 1 linha (Table.Skip)
    df = df.iloc[1:].reset_index(drop=True)
    
    print(f"[DEBUG] Após Skip(1): shape={df.shape}")
    
    # PASSO 3: Promover headers NOVAMENTE (segundo PromoteHeaders)
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    
    print(f"[DEBUG] Após 2º PromoteHeaders: colunas={df.columns.tolist()[:10]}")
    print(f"[DEBUG] Primeiras 3 linhas:\n{df.head(3)}")
    
    # Remover colunas duplicadas
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Renomear colunas principais
    rename_map = {
        'Full Name': 'Colaborator',
        'Rpt Submit Date': 'Submit Date',
        'Entr Transaction Date': 'Transaction Date',
        'Rpt Barcode Id': 'Primary Key',
        'Location Country Description': 'Country Origin',
        'Rpt Custom 4': 'Destination City',
        'Rpt Custom 5': 'Destination Country',
        'Entr Expense Type': 'Expense Type',
        'Journal Amount USD': 'Amount USD',
        'Location Country Description_1': 'Location Country'
    }
    df.rename(columns=rename_map, inplace=True)
    
    # Converter tipos de data
    date_cols = ['Submit Date', 'Transaction Date', 'Filename Date']
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Converter numéricos
    if 'Amount USD' in df.columns:
        df['Amount USD'] = pd.to_numeric(df['Amount USD'], errors='coerce')
    if 'SSO' in df.columns:
        df['SSO'] = pd.to_numeric(df['SSO'], errors='coerce')
    if 'Supervisor Id' in df.columns:
        df['Supervisor Id'] = pd.to_numeric(df['Supervisor Id'], errors='coerce')
    if 'HR Supervisor ID' in df.columns:
        df['HR Supervisor ID'] = pd.to_numeric(df['HR Supervisor ID'], errors='coerce')
    if 'Attendee Count Emp' in df.columns:
        df['Attendee Count Emp'] = pd.to_numeric(df['Attendee Count Emp'], errors='coerce')
    
    # Fill down (Preenchido Abaixo) - todas as colunas que o Power M preenche
    fill_cols = [
        'Colaborator', 'SSO', 'Supervisor Name', 'Supervisor Id', 'Location City',
        'Country Origin', 'HR Supervisor Name', 'HR Supervisor ID', 'Submit Date',
        'Filename Date', 'Transaction Date', 'Destination City', 'Rpt Custom 3',
        'Destination Country', 'Primary Key', 'Rpt Name', 'Expense Type', 'Expense Group',
        'Entr Vendor Description', 'Trip Type', 'AD Type Group', 'Attendee Count Emp',
        'Entr Vendor Name', 'Jrnl Payee Payment Type', 'Jrnl Payer Payment Type',
        'Entr Transaction Type', 'Entr Domestic Foreign', 'Entr Description',
        'Tr Tax Future Use 5', 'Amount USD', 'Channel', 'Entr Country', 'Location Country'
    ]
    
    existing_fill = [c for c in fill_cols if c in df.columns]
    if existing_fill:
        df[existing_fill] = df[existing_fill].ffill()
    
    # Limpar espaços (Valor Substituído)
    if 'Expense Type' in df.columns:
        df['Expense Type'] = df['Expense Type'].astype(str).str.replace(' ', '', regex=False)
    if 'Expense Group' in df.columns:
        df['Expense Group'] = df['Expense Group'].astype(str).str.replace(' ', '', regex=False)
    
    # Remover colunas desnecessárias (replicar Colunas Removidas do Power M)
    cols_to_remove = [
        'External Region', 'Function Type', 'Total Level Accountability Code',
        'Total Level Accountability Desc', 'Q Level Accountability Code',
        'Q Level Accountability Desc', 'R Level Accountability Code',
        'R Level Accountability Desc', 'S Level Accountability Code',
        'S Level Accountability Desc', 'T Level Accountability Code',
        'T Level Accountability Desc', 'U Level Accountability Code',
        'U Level Accountability Desc', 'V Level Accountability Code',
        'V Level Accountability Desc', 'W Level Accountability Code',
        'W Level Accountability Desc', 'X Level Accountability Code',
        'X Level Accountability Desc', 'Y Level Accountability Code',
        'Y Level Accountability Desc', 'Z Level Accountability Code',
        'Z Level Accountability Desc', 'N Level Region Desc',
        'L Level Region Desc', 'J Level Region Desc', 'G Level Region Desc',
        'E Level Region Desc', 'C Level Region Desc', 'A Level Region Desc',
        'Supervisor Name Level 00', 'Supervisor Name Level 01',
        'Supervisor Name Level 02', 'Supervisor Name Level 03',
        'Supervisor Name Level 04', 'Supervisor Name Level 05',
        'Supervisor Name Level 06', 'Supervisor Name Level 07',
        'Supervisor Name Level 08', 'Jrnl Account Code', 'Company Code',
        'Emp Cost Center Org Unit 2', 'Accountability Code', 'Account Detail Type',
        'Enh Alloc Cost Center', 'Emp Cost Center Org Unit 2_2'
    ]
    
    df = df.drop(columns=[c for c in cols_to_remove if c in df.columns])
    
    # Reordenar colunas (Colunas Reordenadas do Power M)
    final_cols_order = [
        'Colaborator', 'Supervisor Name', 'Submit Date', 'Transaction Date',
        'Primary Key', 'SSO', 'Supervisor Id', 'Country Origin', 'Location City',
        'Destination Country', 'Destination City', 'HR Supervisor Name',
        'HR Supervisor ID', 'Filename Date', 'Rpt Custom 3', 'Rpt Name',
        'Expense Type', 'Expense Group', 'Entr Vendor Description', 'Trip Type',
        'AD Type Group', 'Attendee Count Emp', 'Entr Vendor Name',
        'Jrnl Payee Payment Type', 'Jrnl Payer Payment Type', 'Entr Transaction Type',
        'Entr Domestic Foreign', 'Entr Description', 'Tr Tax Future Use 5',
        'Amount USD', 'Channel', 'Entr Country', 'Location Country'
    ]
    
    # Pegar apenas as colunas que existem
    available_cols = [c for c in final_cols_order if c in df.columns]
    result = df[available_cols].copy()
    
    print(f"\n[RESULTADO] Shape final: {result.shape}")
    print(f"[RESULTADO] Primeiras 5 linhas:\n{result.head()}")
    
    return result

In [53]:
# ========================================
# BLOCO 2: T&E 2024
# ========================================

def process_te_2024():
    # Carregar Excel
    df = pd.read_excel(
        r".\OAS Data - Fernanda Cary Supervisor 4-2-2025.xlsx",
        sheet_name="Sheet1"
    )
    
    # Renomear colunas
    rename_map = {
        'Full Name': 'Colaborator',
        'Location Country Description': 'Country Origin',
        'Entr Transaction Date': 'Transaction Date',
        'Rpt Submit Date': 'Submit Date',
        'Rpt Custom 4': 'Destination City',
        'Rpt Custom 5': 'Destination Country',
        'Journal Amount USD': 'Amount USD',
        'Entr Expense Type': 'Expense Type'
    }
    df.rename(columns=rename_map, inplace=True)
    
    # Converter datas
    df['Submit Date'] = pd.to_datetime(df['Submit Date'], errors='coerce')
    df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
    
    # Filtrar ano anterior (2024)
    df = df[df['Submit Date'].dt.year == 2024]
    
    # Limpar espaços
    df['Expense Type'] = df['Expense Type'].str.replace(' ', '')
    df['Expense Group'] = df['Expense Group'].str.replace(' ', '')
    
    # Selecionar colunas
    cols_keep = ['Colaborator', 'SSO', 'Supervisor Name', 'Submit Date', 
                 'Transaction Date', 'Country Origin', 'Location City',
                 'Destination Country', 'Destination City', 'Expense Type',
                 'Expense Group', 'Amount USD', 'Channel']
    
    return df[cols_keep]

In [54]:
# ========================================
# BLOCO 3: COMBINAR T&E
# ========================================

def combine_te_data():
    te_2025 = process_te_2025()
    te_2024 = process_te_2024()
    
    # Combinar
    df = pd.concat([te_2024, te_2025], ignore_index=True)
    
    # Renomear Destination Country para Destination
    df.rename(columns={'Destination Country': 'Destination'}, inplace=True)
    
    # Filtrar Diego após abril 2025
    mask_diego = (
        (df['Colaborator'].str.upper().str.strip() == 'GUEVARA, DIEGO OSWALDO (DIEGO)') &
        (df['Transaction Date'] >= '2025-04-01')
    )
    df = df[~mask_diego]
    
    return df

In [55]:
# ========================================
# BLOCO 4: LIMITS
# ========================================

def load_limits():
    df = pd.read_excel(
        r".\Limits.xlsx",
        sheet_name="Sheet1"
    )
    
    # Remover colunas desnecessárias
    df = df[['Key', 'Despesa', 'Country - Region', 'Country', 'Tipo', 'Moeda', 'Limit']]
    
    # Substituir N/A por NaN e converter Limit para numérico
    df['Limit'] = df['Limit'].replace('N/A', np.nan)
    df['Limit'] = pd.to_numeric(df['Limit'], errors='coerce')
    
    return df

In [56]:
# ========================================
# BLOCO 5: T&L RESERVATIONS
# ========================================

def load_tl_reservations():
    df = pd.read_excel(
        r".\T&L Reservations (CWT) - Imaging.xlsx",
        sheet_name="Sheet1"
    )
    return df

In [ ]:
# ========================================
# BLOCO 6: CONFIRMATION DAYS
# ========================================

def load_confirmation_days():
    df = pd.read_excel(
        r".\Confirmation Days In Advance_LATAM-2026-01-28-16-45-30.xlsx",
        sheet_name="Confirmation Days In Advance_LA",
        skiprows=11
    )
    print("ORIGINAL:")
    print(f"Shape: {df.shape}")
    print(f"Colunas: {df.columns.tolist()}")
    print(f"Duplicadas? {df.columns.duplicated().sum()}")
    print(df.head())
    
    # Remover primeira coluna se vazia
    if df.columns[0] in [0, 'Unnamed: 0'] or pd.isna(df.columns[0]) or 'Unnamed' in str(df.columns[0]):
        df = df.iloc[:, 1:]
    
    print(f"Colunas originais: {df.columns.tolist()}")
    
    # **CRÍTICO: Remover colunas duplicadas ANTES de qualquer operação**
    df = df.loc[:, ~df.columns.duplicated(keep='first')]
    
    print(f"Colunas após remover duplicatas: {df.columns.tolist()}")
    print(f"Shape: {df.shape}")
    
    # Renomear colunas por posição
    col_names = [
        'Technician Name',
        'WO Reference Number', 
        'Work Order Number',
        'Status',
        'Subject',
        'Scheduled Training Start Date Time',
        'Actual Training Start Date',
        'Last Modified Date',
        'Created Date',
        'Created By Full Name'
    ]
    
    # Ajustar se tiver mais/menos colunas
    if len(df.columns) < len(col_names):
        col_names = col_names[:len(df.columns)]
    elif len(df.columns) > len(col_names):
        col_names.extend([f'Extra_{i}' for i in range(len(df.columns) - len(col_names))])
    
    df.columns = col_names
    
    print(f"Colunas após renomear: {df.columns.tolist()}")
    
    # **USAR 'Actual Training Start Date' como a data real**
    # A coluna 'Scheduled Training Start Date Time' tem texto de email!
    
    # Converter datas
    df['Actual Training Start Date'] = pd.to_datetime(df['Actual Training Start Date'], errors='coerce')
    df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')
    df['Last Modified Date'] = pd.to_datetime(df['Last Modified Date'], errors='coerce')
    
    # Extrair GON do Work Order Number (formato: "GON#5522735")
    df['GON'] = df['Work Order Number'].str.extract(r'GON#(\d+)', expand=False)
    
    # Extrair nome após hífen (se houver)
    df['Technician Name'] = df['Technician Name'].astype(str).str.split('-').str[-1].str.strip()
    
    # **IMPORTANTE: Dropar a coluna com email e renomear a coluna com data real**
    df = df.drop(columns=['Scheduled Training Start Date Time'], errors='ignore')
    df.rename(columns={'Actual Training Start Date': 'Scheduled Training Start Date Time'}, inplace=True)
    
    print(f"Colunas finais: {df.columns.tolist()}")
    print(f"Primeiras datas: {df['Scheduled Training Start Date Time'].head()}")
    
    return df

In [58]:
def process_tl_with_metrics():
    # Carregar dados base
    tl = load_tl_reservations()
    conf = load_confirmation_days()
    
    print(f"Primeiras linhas de conf após load:")
    print(conf[['Technician Name', 'GON', 'Scheduled Training Start Date Time', 'Created Date']].head())
    
    # Preparar confirmation com padronização de nomes
    conf['Colaborador'] = conf['Technician Name'].str.upper().str.strip()
    
    # Padronizar nomes (mapeamento completo)
    name_map = {
        'AMANDA': 'AMANDA SARAHI', 'ANDRES': 'ANDRES', 'CARLA': 'CARLA',
        'CESAR': 'CESAR', 'CLAUDIOMAR': 'CLAUDIOMAR', 'CRISTIANE': 'CRISTIANE',
        'DANILO': 'DANILO CHAVES', 'DIANA': 'DIANA', 'DIEGO': 'DIEGO',
        'ENRIQUE': 'ENRIQUE', 'ERIK': 'ERIK', 'FRANCISCO': 'FRANCISCO JAVIE',
        'GABRIELA': 'GABRIELA', 'IBIS': 'IBIS', 'JAEL': 'JAEL SARAI',
        'JAIME': 'JAIME', 'JARDEL': 'JARDEL', 'JAVIER': 'JAVIER',
        'JESUS': 'JESUS DAVID', 'JOHN': 'JOHN', 'JONATHAN': 'JONATHAN ELIA',
        'JOSE': 'JOSE JONAT', 'JUAN': 'JUAN', 'JULIANA': 'JULIANA',
        'LEONARDO': 'LEONARDO', 'LORENA': 'LORENA', 'LUCAS': 'LUCAS',
        'LUCIO': 'LUCIO', 'MARCO': 'MARCO', 'MARIA': 'MARIA MONICA',
        'MAYARA': 'MAYARA', 'MONICA': 'MONICA ARELY', 'OSCAR': 'OSCAR SALVADOR',
        'PABLO': 'PABLO', 'PALOMA': 'PALOMA', 'RENATA': 'RENATA',
        'ROBERTO': 'ROBERTO', 'RODOLFO': 'RODOLFO', 'RODRIGO': 'RODRIGO ADAN',
        'RUDACK': 'RUDACK', 'SANDRA': 'SANDRA', 'SUELLEN': 'SUELLEN',
        'VIVIAN': 'VIVIAN', 'WANDERLEI': 'WANDERLEI', 'YAMIL': 'YAMIL',
        'ELIZABETH': 'ELIZABETH', 'CAROLINE': 'CAROLINE', 'OSCAR': 'OSCAR'
    }
    
    def standardize_name(name):
        if pd.isna(name):
            return name
        first = name.split()[0] if ' ' in name else name
        return name_map.get(first, first)
    
    conf['Colaborador_Padronizado'] = conf['Colaborador'].apply(standardize_name)
    tl['Colaborador'] = tl['Traveler Name'].str.split('/').str[-1].str.upper().str.strip()
    tl['Colaborador_Padronizado'] = tl['Colaborador'].apply(standardize_name)
    
    # Renomear e preparar T&L
    tl.rename(columns={'Air Destination Country': 'Destination'}, inplace=True)
    
    # Converter datas
    tl['Departure Date'] = pd.to_datetime(tl['Departure Date'], errors='coerce')
    tl['Issue Date'] = pd.to_datetime(tl['Issue Date'], errors='coerce')
    
    # Converter datas do conf
    conf['Scheduled Training Start Date Time'] = pd.to_datetime(conf['Scheduled Training Start Date Time'], errors='coerce')
    conf['Created Date'] = pd.to_datetime(conf['Created Date'], errors='coerce')
    
    # Remover nulos
    print(f"\nLinhas TL antes de filtrar: {len(tl)}")
    tl = tl[tl['Departure Date'].notna()].copy()
    print(f"Linhas TL após filtrar: {len(tl)}")
    
    print(f"Linhas conf antes de filtrar: {len(conf)}")
    conf = conf[conf['Scheduled Training Start Date Time'].notna()].copy()
    print(f"Linhas conf após filtrar: {len(conf)}")
    
    if len(conf) == 0:
        print("⚠️ AVISO: Nenhuma data válida. Retornando TL sem métricas.")
        return tl
    
    # ========================================
    # NOVO: DEDUPLICAR GONs mantendo Created Date mais recente
    # ========================================
    print(f"\nLinhas conf antes de deduplicar GONs: {len(conf)}")
    
    # Ordenar por GON e Created Date (mais recente primeiro)
    conf = conf.sort_values(['GON', 'Created Date'], ascending=[True, False])
    
    # Manter apenas o registro mais recente de cada GON
    conf = conf.drop_duplicates(subset='GON', keep='first')
    
    print(f"Linhas conf após deduplicar GONs: {len(conf)}")
    # ========================================
    
    # Filtrar Diego após abril 2025
    tl = tl[~((tl['Colaborador_Padronizado'] == 'DIEGO') & (tl['Issue Date'] >= '2025-04-01'))].copy()
    
    # Ordenar para merge_asof
    tl = tl.sort_values('Departure Date').reset_index(drop=True)
    conf = conf.sort_values('Scheduled Training Start Date Time').reset_index(drop=True)
    
    # Merge com janela de ±3 dias
    merged = pd.merge_asof(
        tl,
        conf[['Colaborador_Padronizado', 'GON', 'Created Date', 'Scheduled Training Start Date Time']],
        left_on='Departure Date',
        right_on='Scheduled Training Start Date Time',
        by='Colaborador_Padronizado',
        tolerance=pd.Timedelta(days=3),
        direction='nearest'
    )
    
    # Calcular métricas
    merged['Advance Notice (Days)'] = (
        merged['Scheduled Training Start Date Time'] - merged['Created Date']
    ).dt.days
    
    merged['Ticket Purchase After Confirmation'] = (
        merged['Issue Date'] - merged['Created Date']
    ).dt.days
    
    # Flags
    merged['Negative Day'] = (merged['Ticket Purchase After Confirmation'] < 0).astype(int)
    merged['Training Before Departure'] = (
        merged['Scheduled Training Start Date Time'] < merged['Departure Date']
    ).astype(int)
    
    # Filtros finais
    merged = merged[
        (merged['Usd Amount'].notna()) &
        (merged['Usd Amount'] > 50) &
        (merged['Negative Day'] == 0) &
        (merged['Training Before Departure'] == 0) &
        (merged['Air Ticket Status'] == 'Active') &
        (merged['GON'].notna()) &
        (merged['GON'] != ' ')
    ]
    
    # Deduplicar por GON no resultado final (mantendo Issue Date mais recente)
    merged = merged.sort_values(['GON', 'Issue Date'], ascending=[True, False])
    merged = merged.drop_duplicates(subset='GON', keep='first')
    
    return merged


In [59]:
# ========================================
# BLOCO 8: CONFIRMATION MERGER
# ========================================

def create_confirmation_merger():
    df = load_confirmation_days()
    
    print(f"Colunas disponíveis no merger: {df.columns.tolist()}")
    
    # Padronizar nomes (mesmo mapeamento)
    name_map = {
        'AMANDA': 'AMANDA SARAHI', 'ANDRES': 'ANDRES', 'CARLA': 'CARLA',
        'CESAR': 'CESAR', 'CLAUDIOMAR': 'CLAUDIOMAR', 'CRISTIANE': 'CRISTIANE',
        'DANILO': 'DANILO CHAVES', 'DIANA': 'DIANA', 'DIEGO': 'DIEGO',
        'ENRIQUE': 'ENRIQUE', 'ERIK': 'ERIK', 'FRANCISCO': 'FRANCISCO JAVIE',
        'GABRIELA': 'GABRIELA', 'IBIS': 'IBIS', 'JAEL': 'JAEL SARAI',
        'JAIME': 'JAIME', 'JARDEL': 'JARDEL', 'JAVIER': 'JAVIER',
        'JESUS': 'JESUS DAVID', 'JOHN': 'JOHN', 'JONATHAN': 'JONATHAN ELIA',
        'JOSE': 'JOSE JONAT', 'JUAN': 'JUAN', 'JULIANA': 'JULIANA',
        'LEONARDO': 'LEONARDO', 'LORENA': 'LORENA', 'LUCAS': 'LUCAS',
        'LUCIO': 'LUCIO', 'MARCO': 'MARCO', 'MARIA': 'MARIA MONICA',
        'MAYARA': 'MAYARA', 'MONICA': 'MONICA ARELY', 'OSCAR': 'OSCAR SALVADOR',
        'PABLO': 'PABLO', 'PALOMA': 'PALOMA', 'RENATA': 'RENATA',
        'ROBERTO': 'ROBERTO', 'RODOLFO': 'RODOLFO', 'RODRIGO': 'RODRIGO ADAN',
        'RUDACK': 'RUDACK', 'SANDRA': 'SANDRA', 'SUELLEN': 'SUELLEN',
        'VIVIAN': 'VIVIAN', 'WANDERLEI': 'WANDERLEI', 'YAMIL': 'YAMIL',
        'ELIZABETH': 'ELIZABETH', 'CAROLINE': 'CAROLINE'
    }
    
    def standardize_name(name):
        if pd.isna(name):
            return name
        first = name.split()[0] if ' ' in name else name
        return name_map.get(first, first)
    
    # Já temos 'Technician Name' processado no load_confirmation_days
    df['Colaborador'] = df['Technician Name'].str.upper().str.strip()
    df['Colaborador_Padronizado'] = df['Colaborador'].apply(standardize_name)
    
    # **CORRIGIDO: Usar o nome correto da coluna**
    cols_to_select = [
        'Colaborador', 
        'Colaborador_Padronizado', 
        'GON',
        'Scheduled Training Start Date Time', 
        'Created Date'
    ]
    
    # Adicionar 'Created By Full Name' se existir
    if 'Created By Full Name' in df.columns:
        cols_to_select.append('Created By Full Name')
    elif 'Created By: Full Name' in df.columns:
        cols_to_select.append('Created By: Full Name')
    
    return df[cols_to_select]

In [60]:
if __name__ == "__main__":
    print("="*60)
    print("PROCESSANDO PIPELINE DE DADOS")
    print("="*60)
    
    # 1. T&E2025
    print("\n[1/8] Processando T&E2025...")
    te_2025 = process_te_2025()
    print(f"✓ T&E2025: {len(te_2025)} linhas, {len(te_2025.columns)} colunas")
    
    # 2. T&E2024
    print("\n[2/8] Processando T&E2024...")
    te_2024 = process_te_2024()
    print(f"✓ T&E2024: {len(te_2024)} linhas, {len(te_2024.columns)} colunas")
    
    # 3. BaseHistorica24a25 (T&E Combined)
    print("\n[3/8] Criando BaseHistorica24a25...")
    base_historica = combine_te_data()
    print(f"✓ BaseHistorica24a25: {len(base_historica)} linhas")
    
    # 4. Limits2025
    print("\n[4/8] Carregando Limits2025...")
    limits = load_limits()
    print(f"✓ Limits2025: {len(limits)} linhas")
    
    # 5. T&L Reservations (CWT) - Imaging
    print("\n[5/8] Carregando T&L Reservations (CWT) - Imaging...")
    tl_reservations = load_tl_reservations()
    print(f"✓ T&L Reservations: {len(tl_reservations)} linhas")
    
    # 6. Confirmation Days In Advance Ap
    print("\n[6/8] Carregando Confirmation Days In Advance Ap...")
    conf_days = load_confirmation_days()
    print(f"✓ Confirmation Days: {len(conf_days)} linhas")
    
    # 7. T&L Reservations - MERGER (com métricas)
    print("\n[7/8] Processando T&L Reservations - MERGER...")
    tl_merger = process_tl_with_metrics()
    print(f"✓ T&L MERGER: {len(tl_merger)} linhas")
    
    # 8. Confirmation Days In Advance MERGER
    print("\n[8/8] Criando Confirmation Days In Advance MERGER...")
    conf_merger = create_confirmation_merger()
    print(f"✓ Confirmation MERGER: {len(conf_merger)} linhas")
    
    print("\n" + "="*60)
    print("EXPORTANDO EXCELS INDIVIDUAIS PARA POWER BI")
    print("="*60)
    
    # Criar pasta de output se não existir
    import os
    output_folder = "PowerBI_Tables"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Exportar cada tabela em Excel individual
    tables = {
        'T&E2025.xlsx': te_2025,
        'T&E2024.xlsx': te_2024,
        'BaseHistorica24a25.xlsx': base_historica,
        'Limits2025.xlsx': limits,
        'TL_Reservations_CWT_Imaging.xlsx': tl_reservations,
        'Confirmation_Days_In_Advance_Ap.xlsx': conf_days,
        'TL_Reservations_MERGER.xlsx': tl_merger,
        'Confirmation_Days_In_Advance_MERGER.xlsx': conf_merger
    }
    
    for filename, df in tables.items():
        filepath = os.path.join(output_folder, filename)
        df.to_excel(filepath, index=False, engine='openpyxl')
        print(f"✓ Exportado: {filename} ({len(df):,} linhas)")
    
    print(f"\n✓ Todos os arquivos salvos em: ./{output_folder}/")
    
    # Resumo final
    print("\n" + "="*60)
    print("RESUMO DAS TABELAS EXPORTADAS")
    print("="*60)
    print(f"1. T&E2025.xlsx                              {len(te_2025):,} linhas")
    print(f"2. T&E2024.xlsx                              {len(te_2024):,} linhas")
    print(f"3. BaseHistorica24a25.xlsx                   {len(base_historica):,} linhas")
    print(f"4. Limits2025.xlsx                           {len(limits):,} linhas")
    print(f"5. TL_Reservations_CWT_Imaging.xlsx          {len(tl_reservations):,} linhas")
    print(f"6. Confirmation_Days_In_Advance_Ap.xlsx      {len(conf_days):,} linhas")
    print(f"7. TL_Reservations_MERGER.xlsx               {len(tl_merger):,} linhas")
    print(f"8. Confirmation_Days_In_Advance_MERGER.xlsx  {len(conf_merger):,} linhas")
    print("="*60)
    print("Pipeline concluído com sucesso! 🎉")
    print(f"Arquivos disponíveis em: ./{output_folder}/")
    print("="*60)

PROCESSANDO PIPELINE DE DADOS

[1/8] Processando T&E2025...
[DEBUG] Shape original: (10002, 80)
[DEBUG] Após 1º PromoteHeaders: colunas=['Full Name', 'SSO', 'Supervisor Name', 'Supervisor Id', 'Location City']
[DEBUG] Após Skip(1): shape=(10000, 80)
[DEBUG] Após 2º PromoteHeaders: colunas=[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
[DEBUG] Primeiras 3 linhas:
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN  001  LB_1201  \
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN  034  LB_1201   
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN  034  LB_1201   
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN  001  LB_1201   

0  MX  Mexico  NaN  CC_111235  0  NaN  NaN  
0  MX  Mexico  NaN  CC_111235  2  NaN  NaN  
1  MX  Mexico  NaN  CC_111235  0  NaN  NaN  
2  MX  Mexico  NaN  CC_111235  2  NaN  NaN  

[3 rows x 80 columns]

[RESULTADO] Shape final: (9999, 0)
[RESULTADO] Primeiras 5 linhas:
Empty DataFrame
Columns: []
Index: [0, 1, 2